<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2018/---%20Day%2024%3A%20Immune%20System%20Simulator%2020XX%20---.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''Immune System:
17 units each with 5390 hit points (weak to radiation, bludgeoning) with an attack that does 4507 fire damage at initiative 2
989 units each with 1274 hit points (immune to fire; weak to bludgeoning, slashing) with an attack that does 25 slashing damage at initiative 3

Infection:
801 units each with 4706 hit points (weak to radiation) with an attack that does 116 bludgeoning damage at initiative 1
4485 units each with 2961 hit points (immune to radiation; weak to fire, cold) with an attack that does 12 slashing damage at initiative 4'''
with open('24.txt') as f: puzzle = f.read()

puzzle = puzzle.replace('(', '').replace(';', '').replace(',', '').replace(')', '')
puzzle = puzzle.split('\n')
puzzle

#
groups = {}
n = 0

i = 1
group_type =  'immune_system'
while i < len(puzzle):

    if puzzle[i] == '':
        i += 2
        group_type =  'infection'

    data = puzzle[i].split(' ')

    # basics
    g = {}
    g['units'] = int(data[0])
    g['hit_points'] = int(data[4])
    g['immune'] = []
    g['weak'] = []
    g['damage'] = int(data[-6])
    g['damage_type'] = data[-5]
    g['initiative'] = int(data[-1])

    # immunity
    if 'immune' in data:
        j = data.index('immune') + 2
        while True:
            g['immune'].append(data[j])
            j += 1
            if data[j] in ['with', 'weak']: break

    # weakness
    if 'weak' in data:
        j = data.index('weak') + 2
        while True:
            g['weak'].append(data[j])
            j += 1
            if data[j] in ['with', 'immune']: break

    # end
    g['group_type'] = group_type
    groups[n] = g
    n += 1

    i += 1

groups

assert len(set( groups[g]['initiative'] for g in groups )) == n # initiative is unique

################################################################################
# PART 1

# helper
def calculate_damage(g0, g1):
    global groups

    d = 0
    if groups[g0]['damage_type'] in groups[g1]['immune']:
        d = 0
    elif groups[g0]['damage_type'] in groups[g1]['weak']:
        d = groups[g0]['effective_power'] * 2
    else:
        d = groups[g0]['effective_power']
    return d


#
iter = 0
while True:

    # print('Round', iter, '\n')

    # calculate effective power
    for g in groups:
        groups[g]['effective_power'] = groups[g]['units'] * groups[g]['damage']
        if groups[g]['effective_power'] == 0: assert groups[g]['units'] == 0

    immune_system_died = all(groups[g]['effective_power'] == 0 for g in groups if groups[g]['group_type'] == 'immune_system')
    infection_died = all(groups[g]['effective_power'] == 0 for g in groups if groups[g]['group_type'] == 'infection')

    if immune_system_died or infection_died:
        break

    # target selection phase: determine order based on effective power
    groups_alive_in_order = [ g for g in groups if groups[g]['effective_power'] != 0 ]
    groups_alive_in_order = sorted(groups_alive_in_order, key = lambda g_id: -groups[g_id]['initiative']) # '-' sign for descending order in initiative
    groups_alive_in_order = sorted(groups_alive_in_order, key = lambda g_id: -groups[g_id]['effective_power']) # '-' sign for descending order in effective power, also using python sorting stability
    # print('Groups in order:')
    # for g in groups_alive_in_order:
    #     print(g, '; effective_power:', groups[g]['effective_power'])
    # print()

    # target selection phase: target selection
    targets_selected_already = []

    for g0 in groups_alive_in_order:
        opponents = [ g1 for g1 in groups_alive_in_order if groups[g0]['group_type'] != groups[g1]['group_type'] and g1 not in targets_selected_already ]

        # print(g0, '; opponents:', opponents)

        target = None
        assumed_damage = 0
        for g1 in opponents:
            a_d = calculate_damage(g0, g1)
            # print(g0, 'would deal', a_d, 'damage to', g1)

            if assumed_damage < a_d:
                assumed_damage = a_d
                target = g1

        #
        assert target not in targets_selected_already, print(target, targets_selected_already)

        if target != None: targets_selected_already.append(target)
        groups[g0]['assumed_damage'] = assumed_damage
        groups[g0]['target'] = target
        # print(g0, 'chooses the following opponent:', target, '\n')

    # print('---\n')

    ### ATTACKING PHASE
    # first, resort the array based on initiative
    groups_alive_in_order = sorted(groups_alive_in_order, key = lambda g_id: -groups[g_id]['initiative'])
    # print('Groups in order:')
    # for g in groups_alive_in_order:
    #     print(g, '; effective_power:', groups[g]['effective_power'])
    # print()

    for g in groups_alive_in_order:

        # group died in combat before its turn
        if groups[g]['units'] == 0:
            continue

        t = groups[g]['target']

        # group does not have a target
        if t == None:
            continue

        killing = groups[g]['assumed_damage'] // groups[t]['hit_points']
        killing = min(groups[t]['units'], killing)
        # print(g, 'kills', killing, 'unit(s) from target group', t)

        # need to update target units, effective power and assumed damage
        groups[t]['units'] -= killing
        groups[t]['effective_power'] = groups[t]['units'] * groups[t]['damage']
        if groups[t]['target'] != None:
            groups[t]['assumed_damage'] = calculate_damage(t, groups[t]['target'])
        # print(t, 'now only deals', groups[t]['assumed_damage'], 'to', groups[t]['target'], '\n')

    # print('End of round', iter, '\n---\n')

    iter += 1
    if iter == 1000: break

# end of combat
assert immune_system_died + infection_died == 1 # only one winner group type
if immune_system_died:
    W = sum( groups[g]['units'] for g in groups if groups[g]['group_type'] == 'infection' )
else:
    W = sum( groups[g]['units'] for g in groups if groups[g]['group_type'] == 'immune_system' )
W